In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [2]:
path = r"D:\Percia_MTech\GUVI\python\Projects\patrol_IQ\data\chicago_crime_cleaned.csv"
df = pd.read_csv(path,low_memory=False)
df

,id,case_number,date,block,iucr,primary_type,description,location_description,arrest,domestic,...,ward,community_area,fbi_code,x_coordinate,y_coordinate,year,updated_on,latitude,longitude,location
0,1859563,g694523,2001-11-18 05:20:00,013xx w 95 st,0610,burglary,forcible entry,residence,0,0,...,22.0,32.0,05,1169173.0,1841801.0,2001,2015-08-17t15:03:40.000,41.721411,-87.655949,", \n(41.721410833, -87.65594939)"
1,1876400,g724352,2001-12-03 10:30:00,003xx n justine st,0810,theft,over $500,street,0,0,...,22.0,32.0,06,1166128.0,1902494.0,2001,2015-08-17t15:03:40.000,41.888025,-87.665375,", \n(41.888024646, -87.665374996)"
2,1937784,hh109913,2002-01-06 11:15:00,031xx n pulaski rd,0841,theft,financial id theft:$300 &under,grocery food store,0,0,...,22.0,32.0,06,1149179.0,1920548.0,2002,2018-02-28t15:56:25.000,41.937912,-87.727149,", \n(41.93791173, -87.727149134)"
3,1893417,g735159,2001-12-08 11:00:00,024xx n lorel av,0460,battery,simple,residence,0,1,...,22.0,32.0,08b,1140352.0,1915640.0,2001,2015-08-17t15:03:40.000,41.924610,-87.759711,", \n(41.924610364, -87.759711178)"
4,1842851,g676728,2001-11-10 03:40:52,076xx s union av,0460,battery,simple,residence,0,1,...,22.0,32.0,08b,1172999.0,1854138.0,2001,2021-09-07t15:41:02.000,41.755182,-87.641572,", \n(41.755181743, -87.641572274)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
750650,2341670,hh639818,2002-09-10 22:26:54,008xx n sedgwick st,1350,criminal trespass,to state sup land,sidewalk,1,0,...,27.0,8.0,26,1173355.0,1906145.0,2002,2018-02-28t15:56:25.000,41.897886,-87.638727,", \n(41.897885763, -87.638726692)"
750651,2333582,hh618240,2002-09-01 01:30:00,002xx w ontario st,2240,liquor law violation,minor misrepresent age,tavern/liquor store,1,0,...,42.0,8.0,22,1174473.0,1904439.0,2002,2018-02-28t15:56:25.000,41.893179,-87.634671,", \n(41.893179495, -87.634671454)"
750652,2216360,hh482086,2002-07-02 10:26:00,030xx e 79th st,0880,theft,purse-snatching,sidewalk,0,0,...,7.0,46.0,06,1197746.0,1853168.0,2002,2018-02-28t15:56:25.000,41.751938,-87.550915,", \n(41.75193817, -87.550915052)"
750653,2262039,hh542961,2002-07-26 17:00:00,061xx n lincoln ave,1310,criminal damage,to property,parking lot/garage(non.resid.),0,0,...,50.0,13.0,14,1153209.0,1940692.0,2002,2018-02-28t15:56:25.000,41.993109,-87.711801,", \n(41.993108988, -87.711800961)"


Temporal feature Engineering

In [9]:
df['date'] = pd.to_datetime(df['date'])

df['hour'] = df['date'].dt.hour
df['day_of_week'] = df['date'].dt.weekday      # 0=Mon
df['month'] = df['date'].dt.month
df['weekend'] = df['day_of_week'].apply(lambda x: 1 if x>=5 else 0)

In [10]:
# seasonal feature
def get_season(month):
    if month in [12,1,2]: return "Winter"
    elif month in [3,4,5]: return "Spring"
    elif month in [6,7,8]: return "Summer"
    else: return "Autumn"

df['season'] = df['month'].apply(get_season)

Geographic feature engineering

In [11]:
df['lat_bin'] = pd.cut(df['latitude'], bins=20, labels=False)
df['long_bin'] = pd.cut(df['longitude'], bins=20, labels=False)

Crime severity score

In [12]:
severity_map = {
    'HOMICIDE': 3,
    'CRIMINAL SEXUAL ASSAULT': 3,
    'ROBBERY': 3,
    'BATTERY': 2,
    'ASSAULT': 2,
    'BURGLARY': 2,
    'THEFT': 1,
    'MOTOR VEHICLE THEFT': 1,
    'DECEPTIVE PRACTICE': 1
}

df['crime_severity'] = df['primary_type'].map(severity_map).fillna(1)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 750655 entries, 0 to 750654
Data columns (total 22 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    750655 non-null  int64  
 1   case_number           750655 non-null  object 
 2   date                  750655 non-null  object 
 3   block                 750655 non-null  object 
 4   iucr                  750655 non-null  object 
 5   primary_type          750655 non-null  object 
 6   description           750655 non-null  object 
 7   location_description  750655 non-null  object 
 8   arrest                750655 non-null  int64  
 9   domestic              750655 non-null  int64  
 10  beat                  750655 non-null  int64  
 11  district              750655 non-null  float64
 12  ward                  750655 non-null  float64
 13  community_area        750655 non-null  float64
 14  fbi_code              750655 non-null  object 
 15  

In [8]:
df['fbi_code'].value_counts()

fbi_code
06     146924
08b    116335
14      82196
08a     80661
18      66779
26      52573
07      35908
05      35136
03      25200
11      23260
04b     21198
04a     13345
15      10359
24       8816
16       6672
17       5341
10       4118
20       4090
01a      3579
02       3519
22       1507
19       1379
09       1308
13        316
12        130
01b         6
Name: count, dtype: int64

In [13]:
# encoding categorical columns
label_cols = ['primary_type', 'location_description', 'season']

le = LabelEncoder()
for col in label_cols:
    df[col] = le.fit_transform(df[col])


In [14]:
# select features for clustering
features = [
    'latitude', 'longitude', 'hour', 'day_of_week', 'month',
    'weekend', 'crime_severity', 'primary_type', 'lat_bin', 'long_bin'
]

X = df[features].copy()

In [15]:
# standardize data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Shape after scaling:", X_scaled.shape)

Shape after scaling: (750655, 10)


In [31]:
df = pd.get_dummies(df, columns=[
    'gender', 'marital_status', 'education', 'employment_type',
    'company_type', 'house_type', 'emi_scenario'
], drop_first=True)

In [16]:
df.to_csv('D:\Percia_MTech\GUVI\python\Projects\patrol_IQ\data\chicago_crime_featured.csv', index=False)

<>:1: SyntaxWarning: invalid escape sequence '\P'
<>:1: SyntaxWarning: invalid escape sequence '\P'
C:\Users\Satheesh\AppData\Local\Temp\ipykernel_10332\3750885696.py:1: SyntaxWarning: invalid escape sequence '\P'
  df.to_csv('D:\Percia_MTech\GUVI\python\Projects\patrol_IQ\data\chicago_crime_featured.csv', index=False)
